Dữ liệu: SMS Spam (5.572 tin nhắn, nhãn ham/spam).
Yêu cầu:

Huấn luyện mô hình phân loại bằng Pipeline TF-IDF + Logistic Regression.
Cross-validation 5 fold: báo macro-F1 và accuracy (trung bình ± độ lệch), so với baseline.
Báo precision/recall/F1 từng lớp và ma trận nhầm lẫn.
Viết 3–5 câu về cách chống rò rỉ.
Viết 3 câu giới hạn.

In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold,cross_validate,cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

URL = ("https://raw.githubusercontent.com/justmarkham/"
       "pycon-2016-tutorial/master/data/sms.tsv")
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "message"])
X_sms, y_sms = sms["message"], sms["label"]               # giữ nhãn chữ cho báo cáo dễ đọc

pipe = Pipeline([("tfidf",TfidfVectorizer()),
                 ("clf",LogisticRegression(max_iter=1000,class_weight="balanced"))])
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
metrics = ["f1_macro","accuracy"]
scores = cross_validate(pipe,X_sms,y_sms,cv=cv,scoring=metrics)
base = cross_validate(DummyClassifier(strategy="most_frequent"), X_sms, y_sms,
                      cv=cv, scoring=metrics)
for name, res in [("mô hình ", scores), ("baseline", base)]:
    f1, acc = res["test_f1_macro"], res["test_accuracy"]
    print(f"{name}: macro-F1 {f1.mean():.3f} +/- {f1.std():.3f} | "
          f"accuracy {acc.mean():.3f} +/- {acc.std():.3f}")

mô hình : macro-F1 0.960 +/- 0.007 | accuracy 0.982 +/- 0.003
baseline: macro-F1 0.464 +/- 0.000 | accuracy 0.866 +/- 0.000


In [ ]:
# Báo cáo từng lớp và ma trận nhầm lẫn
oof = cross_val_predict(pipe,X_sms,y_sms,cv=cv) # dự đoán ngoài fold cho cả 5.572 tin. Mỗi tin được đoán bởi mô hình chưa từng thấy nó
print(classification_report(y_sms,oof,digits=3))
print(pd.DataFrame(confusion_matrix(y_sms,oof,labels=["ham","spam"]),
                   index=["thật ham","thật spam"], columns=["đoán ham", "đoán spam"]))

              precision    recall  f1-score   support

         ham      0.991     0.988     0.989      4825
        spam      0.924     0.940     0.932       747

    accuracy                          0.982      5572
   macro avg      0.957     0.964     0.960      5572
weighted avg      0.982     0.982     0.982      5572

           đoán ham  đoán spam
thật ham       4767         58
thật spam        45        702


In [ ]:
# Bảng kết quả M3 từ results.csv
runs = pd.read_csv("../results/results.csv")              # mọi lần chạy đã ghi
report = runs[["run_id", "split", "macro_f1_mean", "macro_f1_std"]]
report = report.assign(ket_qua=report["macro_f1_mean"].map("{:.3f}".format)
                       + " ± " + report["macro_f1_std"].map("{:.3f}".format))
print(report[["run_id", "split", "ket_qua"]].to_string(index=False))

# 0,803: rò rỉ do chia ngẫu nhiên.
# 0,206: chia đúng theo file.
# 0,063: chỉ dùng tên máy (phép thử đường tắt).
# 0,206: cấu hình A chính thức.
# 0,202: bỏ đường tắt volume, điểm không đổi đáng kể.

         run_id                         split       ket_qua
    leak-random      StratifiedKFold(5,rs=42) 0.803 ± 0.009
     leak-group StratifiedGroupKFold(5,rs=42) 0.206 ± 0.033
host-only-group StratifiedGroupKFold(5,rs=42) 0.063 ± 0.038
     A-baseline StratifiedGroupKFold(5,rs=42) 0.206 ± 0.033
        A-novol StratifiedGroupKFold(5,rs=42) 0.202 ± 0.034


In [ ]:
# Bài viết 11.4 — Đoạn chống rò rỉ